# FRC Match Team Metrics
For a given team, finds all of the teams that they will compete with and orders them by week/event.

## Setup
In your virtual environment, install pandas and matplotlib: 
  `pip install pandas matplotlib`
* If you are using VS Code, it should ask you to install the IPython extensions.
* If this next cell runs with no errors, you are all set.

In [1]:
import util
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

## Functions to get Team and Event names
Some parts of this notebook depend on a team you are interested in.  Other parts of this notebook need to look up the team name.  This section defines a function to look up the team name.  Same things for event name.

In [2]:
def get_team_name(team_id):
    url = f'https://www.thebluealliance.com/api/v3/team/{team_id}'
    return util.call_tba_api(url).json()['nickname']

def get_event_name(event_id):
    url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/simple'
    resp = util.call_tba_api(url).json()
    return str(resp['year']) + ' ' + resp['name']

## Default Team and Event

In [3]:
YEAR = '2025'
TEAM = 'frc6223'

TEAM_NAME = get_team_name(TEAM)

print(TEAM_NAME)

Arsenal of Engineering


## Events for this Team

In [4]:
url = f'https://www.thebluealliance.com/api/v3/team/{TEAM}/events/{YEAR}'
resp = util.call_tba_api(url).json()
df = pd.DataFrame.from_dict(resp, orient='columns') 
# Remove any off season events
df = df[df['event_type'] <= 0]
# Drop columns we don't care about
df = df[['event_code', 'week', 'short_name', 'city', 'state_prov']]
df.sort_values(by='week', ascending=True, inplace=True, ignore_index=True)
del url, resp
df


,event_code,week,short_name,city,state_prov
0,wimu,4.0,Phantom Lakes,Mukwonago,WI
1,wimi,5.0,Wisconsin,West Allis,WI


## Teams at those Events

In [5]:
df_teams = pd.DataFrame(columns=['team', 'nickname', 'city', 'state_prov'])
cnt = 0
for event in df['event_code']:
    # Get the teams
    url = f'https://www.thebluealliance.com/api/v3/event/{YEAR}{event}/teams/simple'
    resp = util.call_tba_api(url).json()
    tmp = pd.DataFrame.from_dict(resp)
    print(f'{len(tmp)}: Teams at {event}')
    cnt = cnt + len(tmp)
    tmp = tmp[['key', 'nickname', 'city', 'state_prov']]
    tmp.rename(columns={'key': 'team'}, inplace=True)
    # del url, resp
    # tmp.drop(columns=['key', 'nickname', 'city', 'state_prov'], inplace=True)
    df_teams = pd.merge(df_teams, tmp, how='outer', on=['team', 'nickname', 'city', 'state_prov'])
    
print(f'{cnt}: Total teams')
print(f'{len(df_teams)}: Unique teams')
del event, tmp, url, resp, cnt
df_teams
    

38: Teams at wimu
48: Teams at wimi
86: Total teams
75: Unique teams


,team,nickname,city,state_prov
0,frc10264,STORM,Menomonie,Wisconsin
1,frc10287,Milwaukee City RC,Milwaukee,Wisconsin
2,frc10430,West Bend,West Bend,Wisconsin
3,frc10522,Sun Prairie School District,Sun Prairie,Wisconsin
4,frc10553,Orange Overdrive,Oregon,Wisconsin
...,...,...,...,...
70,frc930,Mukwonago BEARs,Mukwonago,Wisconsin
71,frc9401,Midas' Mayhem,Troy,Missouri
72,frc9425,Intra Milwaukee,Milwaukee,Wisconsin
73,frc9676,Hub City STEAM,Marshfield,Wisconsin


## Events for each team

In [ ]:
df_events = pd.DataFrame(columns=['event_code', 'week',	'short_name', 'city', 'state_prov', 'team', 'nickname', 'team_city', 'team_state'])
print('Fetching', end='')
for team in df_teams['team']:
    print('.', end='')
    url = f'https://www.thebluealliance.com/api/v3/team/{team}/events/{YEAR}'
    resp = util.call_tba_api(url).json()
    tmp = pd.DataFrame.from_dict(resp, orient='columns') 
    # Remove any off season events
    tmp = tmp[tmp['event_type'] <= 0]
    # Drop columns we don't care about
    tmp = tmp[['event_code', 'week', 'short_name', 'city', 'state_prov']]
    tmp['team'] = team
    tmp['nickname'] = df_teams[df_teams['team'] == team]['nickname'].values[0]
    tmp['team_city'] = df_teams[df_teams['team'] == team]['city'].values[0]
    tmp['team_state'] = df_teams[df_teams['team'] == team]['state_prov'].values[0]
    # print(len(tmp.columns))
    tmp.reset_index(drop=True, inplace=True)
    # print(tmp)
    df_events = pd.concat([df_events, tmp], ignore_index=True)  

df_events['team'] = df_events['team'].astype(int)
df_events.sort_values(by=['week', 'event_code', 'team'], inplace=True, ignore_index=True)
del url, resp, team, tmp
df_events.head(5)

Fetching..........................................................................

## Print Results

In [39]:
event = ''
for index, row in df_events.iterrows():
    if event != row['event_code']:
        event = row['event_code']
        # print(f'{int(row["week"])])}')
        print('----------------------------------------------------------------------')
        print(f"{int(row['week'])}: {row['short_name'].ljust(35)} {row['city']} , {row['state_prov']} ({row['event_code']})")
    print(f"  {row['team'].replace('frc', '').ljust(5)} {row['nickname'].ljust(30)} {row['team_city']}, {row['team_state']}")

----------------------------------------------------------------------
0: Lake Superior                  Duluth , MN (mndu)
  1306  BadgerBOTS                Middleton, Wisconsin
  1714  MORE Robotics             Milwaukee, Wisconsin
  1732  Hilltopper Robotics       Milwaukee, Wisconsin
  4607  C.I.S.                    Becker, Minnesota
  4693  Talk Nerdy to Me          Rockford , Minnesota
  6318  FE Freedom Engineers      Freedom, Wisconsin
  6381  Red Raider Robotics       Sheboygan, Wisconsin
  6421  WarriorBots               Muskego, Wisconsin
  6574  Ferradermis               Whitewater, Wisconsin
----------------------------------------------------------------------
0: Northern Lights                Duluth , MN (mndu2)
  2290  FLYT                      Rockford, Illinois
----------------------------------------------------------------------
0: Regional Monterrey             Monterrey , N.L. (mxmo)
  4635  PrepaTec - Botbusters     Monterrey, Nuevo León
------------------------